unmix them pixels yooo

In [1]:
import os
import math 
from random import shuffle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from math import isnan
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
from scipy.optimize import curve_fit
import randomforest_CHAIN_S2 as rfcc

In [2]:
# Magic function to auto-update imported first-party scripts 
%load_ext autoreload
%autoreload 2

In [149]:

# Define functions

plt.rcParams['font.family'] = 'serif' #sans-serif'

# def plot_multi_result(x_array, 
#                       y_array, 
#                       alpha, 
#                       colour,
#                       metric, 
#                       cmap, 
#                       fig_title, 
#                       x_axis_label, 
#                       y_axis_label, 
#                       stats_values, 
#                       line_1_1, 
#                       sensor_code, 
#                       class_code,
#                       type = "scatter",
#                       h_lines = None, 
#                       save = False, 
#                       output_dir = None,
#                       output_name = None, 
#                       figsize = 3):
#     fig, ax = plt.subplots(x_array.shape[1]//2 + x_array.shape[1]%2, 2, figsize = (10, (x_array.shape[1]//2 + x_array.shape[1]%2)*figsize), sharex = True, sharey = True)
#     fig.tight_layout(pad = 3)
#     if isinstance(y_array, pd.DataFrame):
#         y_array = y_array.values
#     if isinstance(stats_values, pd.DataFrame):
#         stats_values = stats_values.values
    

#     for col in range(x_array.shape[1]):
#         col_name = x_array.columns[col]
#         if isinstance(colour, dict):
#             class_colour = colour.get(col_name, "magenta")
#         else:
#             class_colour = colour
#         if type == "scatter":
#             ax[col//2, col%2].scatter(x_array[col_name], y_array[:,col], alpha = alpha, c = class_colour, cmap = cmap)
#         elif type == "hexbin":
#             hb = ax[col//2, col%2].hexbin(x = x_array[col_name], 
#                                      y = y_array[:, col], 
#                                      gridsize = 50, 
#                                      bins = "log", 
#                                      cmap = cmap)
            
#             fig.colorbar(hb, ax=ax[col//2, col%2], label='counts')

#         ax[col//2, col%2].set_xlabel(f"{x_axis_label} of {col_name.replace("_", " ")}")
#         ax[col//2, col%2].set_ylabel (y_axis_label)
#         if stats_values is not None:
#             ax[col//2, col%2].annotate(f"{metric} = {stats_values[metric][col]:.3f}", xy = (0.05, 0.8), xycoords = "axes fraction")
#         if line_1_1:
#             ax[col//2, col%2].axline((0,0), slope = 1, color = "lightgrey", linestyle = "--")
#         if h_lines is not None:
#             for h in h_lines:
#                 ax[col//2, col%2].axhline(y = h[0], color = h[1], linestyle = ":")
#     if x_array.shape[1] % 2 == 1:
#         ax[-1, -1].set_visible(False)  
#     fig.suptitle(fig_title, y = 1)
    
#     if save: 
#         if not sensor_code or not class_code:
#             ValueError("Sensor code and class code must be provided to save the figure.")
#         if output_dir is None :
#             output_dir = "./"
        
#         os.makedirs(output_dir, exist_ok=True)
#         filename = f"{sensor_code}_{class_code}_{output_name}.svg"
#         filepath = os.path.join(output_dir, filename)
#         fig.savefig(filepath, bbox_inches = "tight")
#     plt.show()
    

def make_categorical_palette(series):
    """
    Create a categorical colour dictionary from unique values in a column,
    using the Tab20 palette. Automatically handles >20 categories by cycling.
    """
    unique_vals = pd.unique(series)

    # Get the Tab20 palette (20 distinct colours)
    base_colors = plt.cm.tab20.colors

    # Cycle through colours if categories > 20
    color_cycle = itertools.cycle(base_colors)

    # Build dictionary
    palette = {val: next(color_cycle) for val in unique_vals}

    return palette

def plot_multi_result(
    x_array,
    y_array,
    alpha,
    colour,
    metric,
    cmap,
    fig_title,
    x_axis_label,
    y_axis_label,
    stats_values,
    line_1_1,
    sensor_code,
    class_code,
    colour_mode = "continuous",
    type="scatter",
    h_lines=None,
    save=False,
    output_dir=None,
    output_name=None,
    figsize=3
):
    # --- Prepare data ---
    if isinstance(y_array, pd.DataFrame):
        y_array = y_array.values
    if isinstance(stats_values, pd.DataFrame):
        stats_values = stats_values.values

    n_cols = x_array.shape[1]
    n_rows = n_cols // 2 + n_cols % 2

    # Create a figure and subplot axes
    fig, ax = plt.subplots(
        n_rows,
        2,
        figsize=(10, n_rows * figsize),
        sharex=True,
        sharey=True
    )
    ax = np.atleast_2d(ax)
    fig.tight_layout(pad=2)

    shared_mappable = None

    # --- Determine colour mapping ---
    if colour_mode == "categorical":
        # mask out nan values
        mask_nan = np.isnan(colour)
        x_array = x_array.iloc[~mask_nan,:]
        y_array = y_array[~mask_nan, :]

        # Build palette dict
        colour_dict = make_categorical_palette(colour)

        # Build per-point colour vector 
        c_values = []
        for v in colour[~mask_nan]:
            c_values.append(colour_dict[v])


    elif colour_mode == "continuous":
        c_values = np.asarray(colour)

    else:
        raise ValueError("Colour mode must be 'categorical' or 'continuous'.")

    # Sublot loop
    for col in range(n_cols):
        col_name = x_array.columns[col]
        row_i, col_i = divmod(col, 2)

        if type == "scatter":
            sc = ax[row_i, col_i].scatter(
                x_array[col_name],
                y_array[:, col],
                alpha=alpha,
                c=c_values,
                cmap=cmap if colour_mode == "continuous" else None
            )
            # if shared_mappable is None:
            shared_mappable = sc

        elif type == "hexbin":
            hb = ax[row_i, col_i].hexbin(
                x=x_array[col_name],
                y=y_array[:, col],
                gridsize=50,
                bins="log",
                cmap=cmap
            )
            # if shared_mappable is None:
            shared_mappable = hb

        # Labels & annotations
        ax[row_i, col_i].set_xlabel(f"{x_axis_label} of {col_name.replace('_', ' ')}")
        ax[row_i, col_i].set_ylabel(f"{y_axis_label} of {col_name.replace('_', ' ')}")

        if stats_values is not None:
            ax[row_i, col_i].annotate(
                f"{metric} = {stats_values[metric][col]:.3f}",
                xy=(0.05, 0.8),
                xycoords="axes fraction"
            )

        # Add 1:1 line if desired
        if line_1_1:
            ax[row_i, col_i].axline((0, 0), slope=1, color="lightgrey", linestyle="--")

        # Add threshold lines if provided
        if h_lines is not None:
            for h in h_lines:
                ax[row_i, col_i].axhline(y=h[0], color=h[1], linestyle=":")

    # Hide last empty subplot if odd number of columns
    if n_cols % 2 == 1:
        ax[-1, -1].set_visible(False)

    # Add colourbar or legend
    if colour_mode == "continuous":
        cbar = fig.colorbar(shared_mappable, ax=ax.ravel().tolist(), shrink=0.9)
        cbar.set_label("Pixel count")

    elif colour_mode == "categorical":
        handles = [
            plt.Line2D([0], [0], marker='o', linestyle='', color=colour_dict[val], label=str(val))
            for val in colour_dict
        ]
        fig.legend(handles=handles, title="Categories", loc="right", bbox_to_anchor=(1.15, 0.5))

    # Add figure title
    fig.suptitle(fig_title, y=1)

    # Save figure
    if save:
        if not sensor_code or not class_code:
            raise ValueError("Sensor code and class code must be provided to save the figure.")

        if output_dir is None:
            output_dir = "./"

        os.makedirs(output_dir, exist_ok=True)
        filename = f"{sensor_code}_{class_code}_{output_name}.svg"
        filepath = os.path.join(output_dir, filename)
        fig.savefig(filepath, bbox_inches="tight")

    plt.show()

def calc_eval_stats(true_values, predicted, include_zeros = False):
    r2_list = []
    mse_list = []
    rmse_list = []
    mae_list = []
    mape_list = []
    all_stats = {}
    predicted = pd.DataFrame(predicted)
    true_values = pd.DataFrame(true_values)
    
    for i in range(true_values.shape[1]):

        x = true_values.iloc[:, i]
        y = predicted.iloc[:, i]
        if not include_zeros:
            x = x[x != 0]
            y = y.loc[x.index]
        r2 = r2_score(x, y)
        mse = mean_squared_error(x, y)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(x, y)
       
        den = x.replace(0, np.nan)

        mape = np.nanmean(np.abs(x - y) / den * 100)

        
        
        # try: 
        #     mape = np.abs(true_values.iloc[:, i] - predicted[:, i]) / true_values.iloc[:, i]*100
        # except ZeroDivisionError:
        #     mape = np.nan
        # clean_mape = [x for x in mape if not isnan(x)]
        # mape = np.mean(clean_mape)
        
        
        r2_list.append(r2)
        mse_list.append(mse)
        rmse_list.append(rmse)
        mae_list.append(mae)
        mape_list.append(mape)
        
    all_stats["R2"] = r2_list
    all_stats["MSE"] = mse_list
    all_stats["RMSE"] = rmse_list
    all_stats['MAE'] = mae_list
    all_stats['MAPE'] = mape_list
    
    return all_stats

def sigmoid(x, L ,x0, k, b):
    y = L / (1 + np.exp(-k*(x-x0))) + b
    return (y)

def fit_best_curve(x, y, criterion = ("R2", "max"), include_zeros = True):
     
    """Fit a set of lines to the data and decide on the best one based on R squared value.
        
    Args
        x (array-like): Independent variable data.
        y (array-like): Dependent variable data.
        criterion (str, str): Tuple of the criterion to evaluate the best fit on and how to choose best fit (min or max criterion value). Default is "R2". Options are MAE, RMSE, MSE, MAPE, R2.
        
    Output
        best_fit_params (tuple): Parameters of the best fit curve.
        all_curve_params (dict): Parameters of all fitted curves.
        best_fit_line (str) : Type of curve that fits best
        all_stats (dict) : Evaluation stats of the fitted curves.
    """
    
    all_curve_params = {}
    crit_ordering = criterion[1]
    criterion = criterion[0]
    
    try:
        p0 = [max(y), np.mean(x), 1, 0] # mandatory initial guess
        popt, _ = curve_fit(sigmoid, x, y, p0, method='trf')
        y_pred_sigmoid = sigmoid(x, *popt)
        all_curve_params = {'sigmoid': popt}

        sigmoid_stats = calc_eval_stats(y, y_pred_sigmoid, include_zeros= include_zeros)
        all_stats = {'sigmoid' : sigmoid_stats}

    except RuntimeError as e:
        print("Sigmoid fit failed, defaulting to linear.")
        all_curve_params = {'sigmoid': None}
        all_stats = {'sigmoid' : None}

   
    # Fit straight line
    b, m = np.polyfit(x, y, 1)
    all_curve_params["linear"] = (b, m)
    y_pred_linear = b*x +m

    linear_stats = calc_eval_stats(y, y_pred_linear, include_zeros = include_zeros)
    all_stats['linear'] = linear_stats
    
    if all_stats['sigmoid'] is None:
        best_fit_line = "linear"
    else:
        # Pick best curve
        sigmoid_crit = sigmoid_stats[criterion][0]
        linear_crit = linear_stats[criterion][0]
        d = {"sigmoid": sigmoid_crit, "linear" : linear_crit}
        if crit_ordering == "max":
            best_fit_line = max(d, key = d.get)
        elif crit_ordering == "min":
            best_fit_line = min(d, key = d.get)
        else:
            ValueError("Criterion ordering must be either 'max' or 'min'")
    best_fit_params = all_curve_params.get(best_fit_line)
    
    return best_fit_params, all_curve_params, best_fit_line, all_stats

def invert_predicted_values(predicted_values, y_test, fitted_curves, all_curves, force_fit = None):
    """ 
    Invert the best fitted curves to back-calculate the expected/adjusted FPC values for any given regression output.
    
    """
    predicted_values = pd.DataFrame(predicted_values, index = y_test.index, columns = y_test.columns)
    predicted_covers = pd.DataFrame(index = y_test.index, columns = y_test.columns)
    
    for col in y_test.columns[:-1]:
        if force_fit is not None:
            if force_fit not in ["sigmoid", "linear"]:
                raise ValueError("Forced fit not recognised.")
            else:
                chosen_fit = force_fit
        
        else:
            if len(fitted_curves[col]) == 4:
                chosen_fit = "sigmoid"
            elif len(fitted_curves[col]) == 2: 
                chosen_fit = "linear"
            else:
                raise ValueError("Fitted curve parameters unexpected size.")
        
        if chosen_fit == "sigmoid": 
            # Access sigmoid params
            L, x0, k, b = all_curves[col][chosen_fit]
            # values = np.array([x if x > b else b for x in predicted_values[:,col] ])
            # Calculate inverse sigmoid for predicted values
            x_result = inverse_sigmoid(predicted_values.loc[:, col], L, x0, k, b)
            x_result = np.clip(x_result, 0, 1)
            
            # Store new predicted values
            predicted_covers[col] = x_result
            
        elif chosen_fit == "linear":
            # Access linear params
            b, m = all_curves[col][chosen_fit]
            # Calculate inverse linear for predicted values
            x_result = (predicted_values.loc[:, col] - m) / b
            x_result = np.clip(x_result, 0, 1)
            
            # Store new predicted values
            predicted_covers[col] = x_result

            
    return predicted_covers

def inverse_sigmoid(y, L, x0, k, b):
    """
    Inverse of sigmoid function.
    Given y, returns x.
    """
    
    y = np.asarray(y, dtype=float)

    # Check if y is in valid range
    if np.any((y - b) <= 0) or np.any((y - b) >= L):
        print("Warning: y values outside valid range (b < y < L+b)")
        
    bad = (y - b <= 0) | (y - b >= L)
    if np.any(bad):
        print("Adjusting bad y values:", y[bad])
        
    # Small epsilon to avoid log(0) or negative arguments
    eps = 1e-12

    # Compute the valid range for y
    y_min = b + eps
    y_max = L + b - eps

    # Clip y to the range
    y = np.clip(y, y_min, y_max)
        
    x = x0 - (1/k) * np.log(L/(y - b) - 1)

    return x

def combine_class_regression(classification_results= pd.DataFrame, 
                             regression_results = pd.DataFrame, 
                             water_predictions = pd.Series,
                             threshold = float):
    overall_results = pd.DataFrame(index = classification_results.index, columns= classification_results.columns)
    for col in regression_results.columns:
        for row in regression_results.index:
            if classification_results.loc[row, col] > threshold:
                overall_results.loc[row, col] = regression_results.loc[row, col]
            else:
                overall_results.loc[row, col] = 0
    overall_results["water"] = water_predictions

    return overall_results

In [193]:
def calc_eval_stats(true_values, predicted, include_zeros = False):
    r2_list = []
    mse_list = []
    rmse_list = []
    mae_list = []
    mape_list = []
    all_stats = {}
    predicted = pd.DataFrame(predicted)
    true_values = pd.DataFrame(true_values)
    
    for i in range(true_values.shape[1]):
        x = true_values.iloc[:, i]
        y = predicted.iloc[:, i]
        
        # Check if column is all zeros
        if (x == 0).all():
            r2_list.append(9999)
            mse_list.append(9999)
            rmse_list.append(9999)
            mae_list.append(9999)
            mape_list.append(9999)
            continue
        
        if not include_zeros:
            x = x[x != 0]
            y = y.loc[x.index]
        
        r2 = r2_score(x, y)
        mse = mean_squared_error(x, y)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(x, y)
       
        den = x.replace(0, np.nan)
        mape = np.nanmean(np.abs(x - y) / den * 100)
        
        r2_list.append(r2)
        mse_list.append(mse)
        rmse_list.append(rmse)
        mae_list.append(mae)
        mape_list.append(mape)
        
    all_stats["R2"] = r2_list
    all_stats["MSE"] = mse_list
    all_stats["RMSE"] = rmse_list
    all_stats['MAE'] = mae_list
    all_stats['MAPE'] = mape_list
    
    return all_stats

In [5]:
# Set persistent notebook parameters
colours = {
    "kelp": "gold",
    "brown_algae": "sienna" ,
    "red_veg": "lightcoral",
    "green_veg": "green",
    "mineral" : "grey",
    "water" : "teal",
    "other" : "grey",
    "macrocystis": "goldenrod",
    "ecklonia": "mediumseagreen" ,
    "durvillaea": "blueviolet",
}
sensor_code = "S2"
class_code = "kbrgm"
plot_output_dir = f"C:/Users/s4770224/Documents/Work/Writing/Figures/Obj2/minerg/{sensor_code}/{class_code}/"

spec_lib_path = "./data/processed/resampled//mine_and_specchio/sub-1-spectra_no-standardization/noisy_Sentinel_2_ABC_resampled.csv"
sim_pix_directory = f"data/mixed_sims/minerg/{sensor_code}/{class_code}/"


# Prepare data for classification

In [6]:
sim_pix = np.load(sim_pix_directory + "development_pixels.npy") 
pixel_fpcs = pd.DataFrame(np.load(sim_pix_directory + "development_pixel_fpcs.npy"))
endmember_indices = np.load(sim_pix_directory + "development_pixel_endmembers.npy")
columns = np.load(sim_pix_directory + "development_pixels_columns.npy", allow_pickle = True)
pixel_fpcs.columns = columns

In [7]:
# Extract PCA components and transform dataset
deco = PCA(min(sim_pix.shape[1], 15))
deco.fit(sim_pix) 
decomposed_mixpix = pd.DataFrame(deco.transform(sim_pix))
comps = deco.components_

In [ ]:
#plot the components

plt.plot(comps[:min(sim_pix.shape[1], 10), :].T, label = range(min(sim_pix.shape[1], 10)))
plt.legend(title = "Component")
plt.xlabel("Band")
plt.ylabel("Component weight")
plt.show()

# Perform presence Absence classification

In [9]:
# divide the dataset into training and testing
x_train, x_test, y_train, y_test = train_test_split(decomposed_mixpix, pixel_fpcs, train_size = 0.5, random_state = 7)
x_train = pd.DataFrame(x_train, index= y_train.index)
x_test = pd.DataFrame(x_test, index = y_test.index)

In [10]:
RF_classifier_params = {
    "n_estimators": 200,
    "max_depth": 20,
    "n_jobs" : -1,
    "min_samples_split": 4,
    "min_samples_leaf": 2,
    "max_features" : "sqrt",
}
RF_regressor_params = {
    "bootstrap" : True,
    "criterion" : 'absolute_error',
    "n_jobs" : -1,
    "oob_score" : False,
    "verbose" : 0,
    "warm_start" : False,
    "max_features" : "sqrt",
    }

In [ ]:
pipe = rfcc.UnmixPixels(x_train, y_train, x_test, y_test, presence_absence_thresh= 0.1)
presence_probas, water_pred = pipe.classify_presence(classifier_params= RF_classifier_params, regressor_params= RF_regressor_params, plot = "water")

In [12]:
# Store classifications and water reg results
train_probas = pd.DataFrame({k : v for k, v in presence_probas["train"].items()})
train_probas.set_index(x_train.index, inplace = True)
test_probas = pd.DataFrame({k : v for k, v in presence_probas["test"].items()})
test_probas.set_index(x_test.index, inplace = True)

In [ ]:
# Plot soft classification results
plotting_params = {
    "x_array": y_test.iloc[:, :-1], 
    "y_array": test_probas, 
    "alpha": 0.2, # p[:,-1],
    "metric" : "MAE", 
    "colour": y_test.iloc[:, -1], #colours, #p[:,-1],
    "cmap" : "cividis", 
    "fig_title" : "Soft classification - probability of class presence",
    "x_axis_label": "Simulated fractional cover",
    "y_axis_label" : "Presence probability", 
    "stats_values" : None,
    "line_1_1" : False, 
    "sensor_code" : sensor_code,
    "class_code" : class_code,
    "h_lines" : [(0.2, "orange"), (0.5, "red")],
    "save" : True,
    "type" : "hexbin",
    "output_dir" : plot_output_dir,
    "output_name" : "soft_classification_testset"
}

plot_multi_result(**plotting_params)  

In [14]:
pd.concat([test_probas, pd.Series(water_pred["test"], name = "water", index = y_test.index), y_test], axis = 1, names = ["predicted","water", "true"]).to_csv(plot_output_dir + "testset_soft_classifications.csv", index = True)

# Filter and regress

In [ ]:
regressor_predictions = pipe.filter_and_regress(plot = "none", regressor_params= RF_regressor_params)

# Store regression results
train_regressions = pd.DataFrame({k : v for k, v in regressor_predictions["train"].items()})
test_regressions = pd.DataFrame({k : v for k, v in regressor_predictions["test"].items()})

In [17]:
# Combine classification and regression results by soft threshold
train_results = combine_class_regression(train_probas, 
                                         train_regressions, 
                                         water_pred["train"], 
                                         0.2)
test_results = combine_class_regression(test_probas, 
                                        test_regressions, water_pred["test"],
                                        0.2)


# Adjust for unity
train_adjusted = train_results.div(train_results.sum(axis = 1), axis = 0)
test_adjusted = test_results.div(test_results.sum(axis = 1), axis = 0)

train_results.replace(np.nan, 0, inplace = True)
test_results.replace(np.nan, 0, inplace = True)
test_adjusted.replace(np.nan, 0, inplace = True)

# Calculate evaluation stats
train_stats = calc_eval_stats(pipe.y_train, train_results, include_zeros= False)
test_stats = calc_eval_stats(pipe.y_test, test_results, include_zeros = False)
test_adjusted_stats = calc_eval_stats(pipe.y_test, test_adjusted, include_zeros = False)

In [18]:
# force Dataframes to have numeric dtypes
train_results = train_results.astype("float64")
test_results = test_results.astype("float64")
test_adjusted = test_adjusted.astype("float64")

In [ ]:
# Plot training regressor results

plotting_params = {
    "x_array": y_train, 
    "y_array": train_results, 
    "alpha": 1, # p[:,-1],
    "metric" : "MAE", 
    "colour":train_results.iloc[:, 0], #p[:,-1],
    "cmap" : "cividis", 
    "fig_title" : "Train pixels - Combined results",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "type" : "hexbin", 
    "stats_values" : train_stats,
    "line_1_1" : True, 
    "sensor_code" : sensor_code,
    "class_code" : class_code,
    "save" : True,
    "output_dir" : plot_output_dir, 
    "output_name": "train_results"
}

plot_multi_result(**plotting_params)  

In [ ]:
# Plot testing regressor and classifier results

plotting_params = {
    "x_array": y_test, 
    "y_array": test_results, 
    "alpha": 1, # p[:,-1],
    "metric" : "MAE", 
    "colour":test_results.iloc[:, -1], #p[:,-1],
    "cmap" : "cividis", 
    "fig_title" : "Dev pixels - Combined results",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "stats_values" : test_stats,
    "type": "hexbin",
    "line_1_1" : True, 
    "sensor_code" : sensor_code,
    "class_code" : class_code,
    "save" : True,
    "output_dir" : plot_output_dir, 
    "output_name": "test_results"
}

plot_multi_result(**plotting_params)  

In [ ]:
# Plot adjusted regressor results

plotting_params = {
    "x_array": y_test, 
    "y_array": test_adjusted, 
    "alpha": 1, # p[:,-1],
    "metric" : "MAE", 
    "colour":test_adjusted.iloc[:, -1], #p[:,-1],
    "cmap" : "cividis", 
    "fig_title" : "Dev pixel - Adjusted random forest results",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "stats_values" : test_adjusted_stats,
    "type" : "hexbin", 
    "line_1_1" : True, 
    "sensor_code" : sensor_code,
    "class_code" : class_code,
    "save" : False,
    "output_dir" : plot_output_dir, 
    "output_name": "Dev_adjusted"
}

plot_multi_result(**plotting_params)  

In [ ]:
# Check out the false positives

# Inspect false positive distributions
true_zeros = pd.DataFrame(columns = ["class", "predicted_FPC", "other_brown"])
for col in y_test.columns:
    zeros = y_test[y_test[col] == 0]
    brown = zeros.iloc[:, :-2].sum(axis = 1)
    class_fpc = test_adjusted.loc[zeros.index, col]
    class_name = pd.Series([col for x in brown], index = class_fpc.index)
    class_zeros = pd.concat([class_name, class_fpc, brown], axis =1).rename(columns = {0: "class", col: "predicted_FPC", 1: "other_brown"})
    true_zeros = pd.concat([true_zeros, class_zeros], axis = 0)

# Plot false positives

if class_code == "bo_genus":
    true_zeros["numeric_class"] = true_zeros["class"].factorize()[0]
    jittered_x = true_zeros["numeric_class"] + np.random.uniform(-0.3, 0.3, size=len(true_zeros))
    plt.figure(figsize=(12, 12))
    sns.scatterplot(
        x=jittered_x,
        y=true_zeros["predicted_FPC"],
        hue=true_zeros["other_brown"],
        palette="cividis",
        alpha=0.5,
    )    
    plt.xticks(ticks = range(min(true_zeros["numeric_class"]), max(true_zeros["numeric_class"])+1), labels = true_zeros["class"].unique(), rotation = 45)

else:
    plt.figure(figsize= (4,3))
    sns.stripplot(data = true_zeros, x= "class", y = "predicted_FPC", hue = "class", alpha  = 0.2, jitter = 0.35, linewidth = 0)
    plt.xticks(rotation = 45)
    plt.title("Predicted FPC for true zero cover samples")
    plt.savefig(f"{plot_output_dir}/{sensor_code}_{class_code}__minerg_false_positives.svg")
    plt.title("Predicted FPC for true zero cover samples")

plt.savefig(f"{plot_output_dir}/{sensor_code}_{class_code}_false_positives.svg")
plt.show() 

# Fitting curves and inverting

In [ ]:
fitted_curves = {}
all_stats = {}
all_curves = {}
criterion = ("MAE", "min")
regs = pd.concat([test_regressions, 
                  pd.Series(water_pred["test"], name = "water", index = y_test.index),], axis = 1)
for col in y_test.columns:
    colour = colours.get(col, "magenta")
    best_fit_params, all_curve_params, best_fit_line, curve_stats = fit_best_curve(pipe.y_test.loc[:, col], 
        regs.loc[:, col],
        criterion = criterion, 
        include_zeros = False)
    print(f"Best fit curve is a {best_fit_line.capitalize()} for {col.replace('_', '' )} with parameters {best_fit_params}")
    fitted_curves[col] = best_fit_params
    all_stats[col] = curve_stats
    all_curves[col] =  all_curve_params

In [ ]:
# Plot fitted curves and regression results

fig, ax = plt.subplots(y_test.shape[1]//2 + y_test.shape[1]%2, 2, figsize = (8,12), sharex = True, sharey = True) 
fig.tight_layout(pad = 3)

for col in range(y_test.shape[1]):
    x = y_test[y_test.iloc[:, col] > 0 ]
    y = np.array(test_results[y_test.iloc[:,col] >0])
    col_name = x.columns[col]
    
    class_curves = all_curves.get(col_name)
    class_stats = all_stats.get(col_name)
    x_linspace = np.linspace(np.min(x), np.max(x), 500)
    
    if class_curves.get("sigmoid") is None:
        y_linspace = {
        "linear" : all_curve_params["linear"][0]*x_linspace + class_curves["linear"][1]
        }
    else:
        y_linspace = {
            "sigmoid": sigmoid(x_linspace, *class_curves.get("sigmoid")),"linear" : all_curve_params["linear"][0]*x_linspace + class_curves["linear"][1]
        }
    
    ax[col//2, col%2].scatter(x[col_name], y[:,col], alpha = 0.3, c = colours.get(col_name, "gray"))
    ax[col//2, col%2].set_xlabel(f"Simulated {col_name.replace("_", " ")} FPC")
    ax[col//2, col%2].set_ylabel ("Predicted FPC")
    ax[col//2, col%2].axline((0,0), slope = 1, color = "darkslategrey", alpha = 1, linestyle = "--")
    if class_curves.get("sigmoid") is not None:
        ax[col//2, col%2].plot(x_linspace, y_linspace["sigmoid"], '--', color = "red", label='Fitted curve - Sigmoid')
    ax[col//2, col%2].plot(x_linspace, y_linspace["linear"], ':', color = "slateblue", label='Fitted curve - Linear')

    if class_curves.get("sigmoid") is not None:
        ax[col//2, col%2].annotate(f"{criterion[0].upper()}: \nSigmoid: {class_stats["sigmoid"][criterion[0]][0]:.4f} \nLinear: {class_stats["linear"][criterion[0]][0]:.4f}", xy = (0.05, 0.8), xycoords = "axes fraction", fontsize = 10,)
    else:
        ax[col//2, col%2].annotate(f"{criterion[0].upper()}: \nSigmoid: undefined \nLinear: {class_stats["linear"][criterion[0]][0]:.4f}", xy = (0.05, 0.8), xycoords = "axes fraction", fontsize = 10,)
        
    if x.shape[1] % 2 == 1:
        ax[-1, -1].set_visible(False)  
    fig.suptitle("Fitting curves to regression results", y = 1)

plt.savefig(f"{plot_output_dir}_{sensor_code}_{class_code}__minerg_fitting_curves.svg")    
plt.show()


In [171]:
# Invert and adjust for totality
inverted_predicted = invert_predicted_values(np.array(test_results),
                                             y_test, fitted_curves,
                                             all_curves,
                                             force_fit= None)

inverted_predicted["water"] = water_pred["test"]
adjusted_inverted = inverted_predicted.div(inverted_predicted.sum(axis = 1), axis = 0)
trues = pipe.y_test.replace(np.nan, 0)
inv_adj_stats = calc_eval_stats(trues, adjusted_inverted, include_zeros= False)

In [ ]:
# Calculate the extra bloat and deficit caused by false classifications
negs = pipe.y_test == 0
posi = pipe.y_test >0
positives = test_probas >= 0.2
negatives = test_probas< 0.2
false_posi = negs * positives
false_negs = posi * negatives
false_posi_bloat = test_results*false_posi
false_negs_deficit = test_regressions * false_negs

bloat = false_posi_bloat.sum()/pipe.y_test.sum() *100
deficit = false_negs_deficit.sum()/pipe.y_test.sum() *100
pd.DataFrame([bloat, deficit], index = ["bloat", "deficit"], columns = pixel_fpcs.columns).to_csv(f"{plot_output_dir}bloat_deficit.csv", index = True)
print(bloat)


In [ ]:
plotting_params = {
    "x_array": y_test, 
    "y_array": adjusted_inverted, 
    "alpha": 0.3, #y_test.iloc[:,-1],
    "colour": y_test.iloc[:,-1],
    "metric" : "MAE",
    "cmap" : "cividis", 
    "fig_title" : f"Dev set - Adjusted, {best_fit_line} inverted regressions",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "stats_values" : inv_adj_stats, 
    "line_1_1" : True,
    "type": "hexbin",
    "sensor_code" : sensor_code,
    "class_code" : class_code, 
    "save" : True,
    "output_dir" : "./plots/regression/evaluation/", 
    "output_name": f"dev_{best_fit_line}_inv_adj_hexbin"
}
plot_multi_result(**plotting_params)

In [ ]:
plt.hexbin(x = pipe.y_test.iloc[:, :2].sum(axis = 1), y = adjusted_inverted.iloc[:, :2].sum(axis = 1), gridsize = 50, bins = "log", cmap = "cividis")
plt.title("Combined Kelp and Brown algae FPC\nAdjusted, inverted regression results")
plt.show()

# Run Chained classification-regression on new data

In [ ]:
# Follow normal workflow for unseen data
new_sim_pix = np.load(sim_pix_directory + "unseen_pixels.npy")
new_pixel_fpcs = pd.DataFrame(np.load(sim_pix_directory + "unseen_pixel_fpcs.npy"))
new_endmember_indices = np.load(sim_pix_directory + "unseen_pixel_endmembers.npy")
new_pixel_fpcs.columns = columns

# Decompose new pixels into the PCA components
deco_new_sim_pix = pd.DataFrame(deco.transform(new_sim_pix))

# Unmix using the classifier and regressor pipeline
new_pixel_classifications, new_water, new_pixel_regressions = pipe.unmix_new_data(deco_new_sim_pix, columns)

# Make any pixel not classified as present have zero cover
new_test_results = combine_class_regression(new_pixel_classifications, new_pixel_regressions, new_water, 0.2)
new_adj_results = new_test_results.div(new_test_results.sum(axis = 1), axis = 0)

# Calculate evaluation stats
new_stats = calc_eval_stats(new_pixel_fpcs, new_test_results, include_zeros= False)

# Invert and adjust for totality
new_test_inverted = invert_predicted_values(np.array(new_test_results),
                                            new_pixel_fpcs, 
                                            fitted_curves, 
                                            all_curves, 
                                            force_fit = "linear")

# Overwrite water results to be the initial regressed values
new_test_inverted["water"] = new_test_results["water"]

# Bound predictions to 0-1
new_test_inverted.clip(lower = 0, upper = 1, inplace = True)

# Aread adjust
new_inverted_adjusted = new_test_inverted.div(new_test_inverted.sum(axis = 1),
                                              axis = 0)

# Calculate stats
new_inv_adj_stats = calc_eval_stats(new_pixel_fpcs, 
                                    new_inverted_adjusted, 
                                    include_zeros = False)

In [ ]:
# Plot  results coloured by class FPC
col = -1
plotting_params = {
    "x_array": new_pixel_fpcs, 
    "y_array": new_inverted_adjusted, 
    "alpha": 0.5, #new_p[:,-1],
    "colour": new_pixel_fpcs.iloc[:, col], 
    "metric": "MAE",
    "cmap" : "cividis", 
    "fig_title" : f"Unseen pixels - {best_fit_line} inverted, area adjusted results",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "stats_values" : new_inv_adj_stats,
    "line_1_1" : True, 
    "sensor_code" : sensor_code,
    "class_code" : class_code,
    "type" : "hexbin", 
    "save" : True,
    "figsize" : 2.5, 
    "output_dir" : "./plots/regression/evaluation/", 
    "output_name": f"inv_adj_unseen_{new_pixel_fpcs.columns[col]}_fpc"
}
plot_multi_result(**plotting_params)  

In [179]:
#re-calc stats with zeros
train_stats_zeros = calc_eval_stats(pipe.y_train, train_results, include_zeros= True)
test_stats_zeros = calc_eval_stats(pipe.y_test, test_results, include_zeros = True)
test_adjusted_stats_zeros = calc_eval_stats(pipe.y_test, test_adjusted, include_zeros = True)
inv_adj_stats_zeros = calc_eval_stats(pipe.y_test, adjusted_inverted, include_zeros= True)
new_stats_zeros = calc_eval_stats(new_pixel_fpcs, new_test_results, include_zeros= True)
new_adj_stats_zeros = calc_eval_stats(new_pixel_fpcs, new_adj_results, include_zeros = True)
new_inv_adj_stats_zeros = calc_eval_stats(new_pixel_fpcs, new_inverted_adjusted, include_zeros = True)


In [ ]:
all_eval_stats = pd.concat([pd.DataFrame(train_stats),
                            pd.DataFrame(test_stats),
                            pd.DataFrame(test_adjusted_stats),
                            pd.DataFrame(inv_adj_stats),
                            pd.DataFrame(new_stats),
                            pd.DataFrame(new_inv_adj_stats),
                            pd.DataFrame(train_stats_zeros),
                            pd.DataFrame(test_stats_zeros),
                            pd.DataFrame(test_adjusted_stats_zeros),
                            pd.DataFrame(inv_adj_stats_zeros),
                            pd.DataFrame(new_stats_zeros),
                            pd.DataFrame(new_adj_stats_zeros),
                            pd.DataFrame(new_inv_adj_stats_zeros),], axis = 0, ignore_index= True, names=[
                                "Dev Train",
                                "Dev Test",
                                "Dev Test Area Adjusted",
                                "Dev Test Adjusted, Inverted",
                                "Unseen",
                                "Unseen Adjusted, Inverted",
                            ])

all_eval_stats.to_csv(f"./results/{sensor_code}_{class_code}_{best_fit_line}_all_eval_statistics_SENSITIVITY.csv", index = True)

print(f"Inverted and Area adjusted: \n {pd.DataFrame(new_inv_adj_stats)}\n \n")


# Calculate curve inversion stats

In [181]:
inversion_results = {
    "test_results": None,
    "new_test_results": None
}
inversion_stats = {
    "test_results": None,
    "new_test_results": None
}

data_dict = {
    "test_results": {
        "true": y_test,
        "pred": test_results,
    },
    "new_test_results": {
        "true": new_pixel_fpcs,
        "pred": new_test_results,
    },}

In [ ]:
for key in data_dict.keys():
    data = data_dict[key]["pred"]
    data_label = key
    col_results = {}
    for col in data.columns:
        class_curves = all_curves.get(col)
        target_col = data.loc[:, col]
        curve_results = {}
        for curve in class_curves.items():
            if curve[1] is None:
                pass
            elif len(curve[1]) == 4:
                # Access sigmoid params
                L, x0, k, b = curve[1]
                # Calculate inverse sigmoid for predicted values
                x_result = inverse_sigmoid(target_col, L, x0, k, b)
                x_result = np.clip(x_result, 0, 1)
                
                # Store new predicted values
                curve_results[curve[0]] = x_result
                
            elif len(curve[1]) == 2:
                # Access linear params
                b, m = curve[1]
                # Calculate inverse linear for predicted values
                x_result = (target_col - m) / b
                x_result = np.clip(x_result, 0, 1)
                
                # Store new predicted values
                curve_results[curve[0]] = x_result

            col_results[col] = curve_results
    inversion_results[data_label] = col_results


In [184]:
for dataset in inversion_results.keys():
    pred_data = inversion_results[dataset]
    dataset_stats = {}
    for target in pred_data.keys():
        target_pred = pred_data[target]
        curve_stats = {}
        y_true = data_dict[dataset]["true"].loc[:, target]
        y_true.replace(np.nan, 0, inplace = True)
        for curve in target_pred.keys():
            y_pred = pd.DataFrame(target_pred[curve], index = y_true.index)
            stats = calc_eval_stats(y_true, y_pred, include_zeros = False)
            curve_stats[curve] = stats
        dataset_stats[target] = curve_stats
    inversion_stats[dataset] = dataset_stats
    
inversion_comps = pd.DataFrame.from_dict({(i, j,): inversion_stats[i][j] 
                           for i in inversion_stats.keys() 
                           for j in inversion_stats[i].keys()},
                       orient='index')

inversion_comps.to_csv(f"./results/{sensor_code}_{class_code}_inversion_evaluation_statistics.csv", index = True)

# Sensitivity investigating

In [192]:
# Determine if sensitivity will be tested
if class_code in ["kbrgm", "Class"]:
    sensitivity = True
    if class_code == "kbrgm":
        sensi_path = f"data/mixed_sims/feb2026/{sensor_code}/SENSITIVITY_kbrgm/"
        genus_test_columns = ["kelp", "brown_algae", "red_veg", "green_veg", "mineral", "water"]
        marker_column = "kelp"
    else:
        sensi_path = f"data/mixed_sims/feb2026/{sensor_code}/SENSITIVITY_macro/"
        genus_test_columns = ["macrocystis", "ecklonia", "undaria", "durvillaea"]
        marker_columns = "macrocystis"
else:
    sensitivity = False

In [199]:
# Workflow for sensitivity
if sensitivity: 
    new_sim_pix = np.load(sensi_path + "unseen_pixels.npy")
    new_pixel_fpcs = pd.DataFrame(np.load(sensi_path + "unseen_pixel_fpcs.npy"))
    new_endmember_indices = np.load(sensi_path  + "unseen_pixel_endmembers.npy")
    new_pixel_fpcs.columns = np.load(sensi_path + "development_pixels_columns.npy", allow_pickle = True)

    temp_fpcs = pd.DataFrame(0.00, index = new_pixel_fpcs.index, columns = genus_test_columns)
    temp_fpcs.update(new_pixel_fpcs)
    new_pixel_fpcs = temp_fpcs

In [ ]:

# Decompose pixels into PCA components
if sensitivity: 
    deco_new_sim_pix = pd.DataFrame(deco.transform(new_sim_pix))

    # Unmix using the classifier and regressor pipeline
    new_pixel_classifications, new_water, new_pixel_regressions = pipe.unmix_new_data(deco_new_sim_pix, columns)

    # Make any pixel not classified as present have zero cover
    new_test_results = combine_class_regression(new_pixel_classifications, new_pixel_regressions, new_water, 0.2)

    # Calculate evaluation stats
    new_stats = calc_eval_stats(new_pixel_fpcs, new_test_results, include_zeros= False)

    # Area adjust and calculate stats
    new_adj_results = new_test_results.div(new_test_results.sum(axis = 1), axis = 0)
    new_adj_stats = calc_eval_stats(new_pixel_fpcs, new_adj_results, include_zeros = False)

    # Invert and adjust for totality
    new_test_inverted = invert_predicted_values(np.array(new_test_results), new_pixel_fpcs, fitted_curves, all_curves, force_fit = "linear")
    new_test_inverted["water"] = new_test_results["water"]
    new_test_inverted.clip(lower = 0, upper = 1, inplace = True)
    new_inverted_adjusted = new_test_inverted.div(new_test_inverted.sum(axis = 1), axis = 0)
    new_inv_adj_stats = calc_eval_stats(new_pixel_fpcs, new_inverted_adjusted, include_zeros = False)

In [ ]:
# If running improper workflow

if sensitivity: 
    # Check number of high false positives
    pred_above_10p = ((new_inverted_adjusted.iloc[:, 1:-1] > 0.00).sum(axis = 0))
    print(f"Number of pixels predicted above 10% FPC:\n{pred_above_10p}")


    # Plot the distribution of predicted FPC for the true zero cover samples to inspect false positives
    fig, ax = plt.subplots(1,4, sharex = True, sharey = True, figsize = (6,2))


    ax[0].hist(new_inverted_adjusted.iloc[:, 1], bins = 50, color = "sienna")
    ax[0].set_ylim(0, new_pixel_fpcs.shape[0]//10)
    ax[0].set_xlim(0, 1)
    ax[0].set_ylabel("Pixel Count")
    ax[1].hist(new_inverted_adjusted.iloc[:, 2], bins = 50, color = "lightcoral")
    ax[2].hist(new_inverted_adjusted.iloc[:, 3], bins = 50, color = "green")
    ax[3].hist(new_inverted_adjusted.iloc[:, 4], bins = 50, color = "grey")
    ax[3].set_xlabel("Predicted FPC")

    fig.suptitle("False positive predicted FPC")
    fig.savefig(f"{plot_output_dir}false_positive_fpc_histograms.svg", bbox_inches = "tight")

    plt.show()

## Check out the endmembers going into each pixel

In [ ]:
#Import spectra

if sensitivity: 
    spectra = pd.read_csv("data/processed/resampled/minerg/feb2026/good/noisy_Sentinel_2_ABC_resampled.csv", index_col = 0)

    # Plot the endmembers of a chosen class
    sensitive_indices = [int(x) for x in pd.DataFrame(new_endmember_indices).iloc[:, 0].unique() if not isnan(x)]

    plt.plot(spectra.loc[sensitive_indices, :].T)
    plt.legend(sensitive_indices)
    plt.xticks(rotation = 45)
    plt.xlabel("Sensor Band")
    plt.ylabel("Reflectance")
    plt.show()

Let's look at the endmembers going into false positive detections

In [ ]:
if sensitivity: 
    # check the column of false positives to check
    col = 4

    # How many false positives is considered too many?
    b = 60

    false_posi = new_inverted_adjusted.iloc[:, col] > 0
    false_posi_counts = pd.DataFrame(new_endmember_indices).loc[false_posi, 0].value_counts()
    outlying_bands = [int(x) for x in false_posi_counts[false_posi_counts > b].index.values]

    print(f"The endmembers contributing to more than {b} false positive {pixel_fpcs.columns[col]} detections each are: {outlying_bands}")


In [214]:
# Define endmember colour dictionary

if sensitivity: 
    possible_colours = ["skyblue" , "m", "palegreen", "coral", "peru", "rebeccapurple", "khaki",  "palegoldenrod","mediumaquamarine", "slateblue", "darkgoldenrod", "olivedrab", "mediumvioletred", "cadetblue", "indigo", "crimson",  "springgreen",  "lightcoral",  "sandybrown", "mediumturquoise", "darkkhaki", "deepskyblue", "tomato", "brown", "violet", "orange",  "limegreen", "mediumpurple", "darkseagreen","bisque", "yellowgreen",  "tan", "sienna","lightcoral", "aquamarine", "slateblue", "mediumorchid",  "gold", "palevioletred"]

    all_colours = {sensitive_indices[i] : possible_colours[i] for i in range(len(sensitive_indices))}


In [ ]:
# False positives investigation plot

if sensitivity: 
    #New figure where the false positives are coloured by their contributing endmembers, to see if there are any patterns in which endmembers are contributing to false positives. The first two panels will be the kelp and water, with most endmembers in grey. Then the four remaining panels are false positives with the colours assigned by endmember and consistent throughout. 

    # Beside this, a second plot with the endmember spectra using the same colours. Most are grey, some are coloured. This will show that spectral diversity leading to errors. 

    # the level of false positives mostly attributable to the  grey points can be said to not be caused by limited spectral representation. This would be attributable to the model itself lacking sensitivity. 

    #Do this twice once for kbrgm, once for kelp genera, hopefully there is less of a distinction in spectra meaning more of the error comes from the model.

    # How many false positives is considered too many?
    b = 75

    fig, ax = plt.subplots(4, 4, figsize = (14, 12), sharey = "col" )

    for row in range (4):
        
        # check the column of false positives to check
        col = row+1



        false_posi = new_inverted_adjusted.iloc[:, col] > 0
        false_posi_counts = pd.DataFrame(new_endmember_indices).loc[false_posi, 0].value_counts()
        outlying_bands = [int(x) for x in false_posi_counts[false_posi_counts > b].index.values]

        endmem_colours = { k : all_colours.get(k) for k in outlying_bands}
        
        # Split indices for this row
        grey_indices = [idx for idx in sensitive_indices if idx not in endmem_colours]
        coloured_indices = [idx for idx in sensitive_indices if idx in endmem_colours]
        
        for idx in grey_indices+ coloured_indices: 
            
            indices_array = new_endmember_indices[:, 0]

            # Build colour array for this row
            colours_array = np.array([endmem_colours.get(ind, "grey") for ind in indices_array])

            # Create mask
            grey_mask = colours_array == "grey"
            colour_mask = ~grey_mask

            # Reorder indices: greys first, coloured second
            ordered_indices = np.concatenate([
                indices_array[grey_mask],
                indices_array[colour_mask]
            ])

            # Reorder y values the same way
            y_values = new_inverted_adjusted.iloc[:, col].values
            ordered_y = np.concatenate([
                y_values[grey_mask],
                y_values[colour_mask]
            ])
            # Reorder colours
            ordered_colours = np.concatenate([
                colours_array[grey_mask],
                colours_array[colour_mask]
            ])

        
            ax[row, 0].scatter(
                        x = new_pixel_fpcs.loc[new_endmember_indices[:, 0] == idx, "kelp"],                       
                        y = new_inverted_adjusted.loc[new_endmember_indices[:, 0] == idx, "kelp"], 
                        color = endmem_colours.get(idx, "grey"), 
                        label = idx, 
                        alpha = 0.2 if endmem_colours.get(idx, "grey") == "grey" else 0.6,
                        linewidths = 0,
                        s = 15
                        )
            ax[row, 1].scatter(
                        x = new_pixel_fpcs.loc[new_endmember_indices[:, 0] == idx, "water"],                       
                        y = new_inverted_adjusted.loc[new_endmember_indices[:, 0] == idx, "water"], 
                        color = endmem_colours.get(idx, "grey"), 
                        label = idx, 
                        alpha = 0.2 if endmem_colours.get(idx, "grey") == "grey" else 0.6,
                        linewidths = 0,
                        s = 15
                        )
            
            ax[row, 2].scatter(x=[str(round(x,0)) for x in ordered_indices],
                            y=ordered_y,
                            c=ordered_colours,
                            s=4,linewidths=0
                            )
            ax[row, 2].hlines(ordered_y.mean()+2*ordered_y.std(), 
                            xmin = 0, 
                            xmax = len(sensitive_indices), 
                            colors = "red", 
                            linestyles = "--", 
                            linewidth = 0.6
                            )
            ax[row, 2].hlines(ordered_y.mean()+ordered_y.std(), 
                            xmin = 0, 
                            xmax = len(sensitive_indices), 
                            colors = "black", 
                            linestyles = ":", 
                            linewidth = 0.6)
            ax[row, 2].set_xticks([])

            ax[row, 3].plot(spectra.loc[idx, :].T, 
                    color = endmem_colours.get(idx, "grey"), 
                    label = idx,
                    alpha = 0.25 if endmem_colours.get(idx, "grey") == "grey" else 1)
            plt.xticks(rotation = 45)
    plt.savefig(f"{plot_output_dir}false_positive_endmember_investigation.svg", bbox_inches = "tight")
    plt.savefig(f"{plot_output_dir}false_positive_endmember_investigation.png", bbox_inches = "tight")
    plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (8,4), sharey = True, sharex = True)
ax[0].scatter(x = new_pixel_fpcs.iloc[:, 0], y = new_test_results.iloc[:, 0], color = "darkgoldenrod", alpha = 0.2)
ax[1].scatter( x = new_pixel_fpcs.iloc[:,0], y = new_inverted_adjusted.iloc[:,0:2].sum(axis = 1), color = "sienna", alpha = 0.2)
plt.savefig(f"{plot_output_dir}combined_kelp_browns_comparison.svg", bbox_inches = "tight")
plt.show()